# Agentic RAG with OpenAI function calling

In this notebook, we will:

1. demonstrate a limitation of direct RAG retrieval;
2. expose the FAQ search function as a tool;
3. let the model formulate a search query;
4. execute the application-owned search function in Python;
5. return the result to the model; and
6. calculate the API cost of the complete interaction.

## 1. Set up the OpenAI client

`openai_client` is the Python SDK client that sends requests to the OpenAI API. It is not itself a language model. Each call to `openai_client.responses.create(...)` selects the remote model using the request’s `model` argument.

For requests made through `RAGBase`, the model defaults to `gpt-5.4-mini`. This can be overridden by passing a different `model` value when constructing the assistant.

In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

## 2. Build the FAQ index

`ingest.py` provides helpers for loading the FAQ documents and creating a search index. In this notebook, `build_index(documents)` creates an in-memory `MinSearch` index. Other indexing backends—such as `SQLitesearch` or `Elasticsearch`—can be assigned to `index` instead, provided they expose the compatible `search()` interface expected by `RAGBase`, either directly or through an adapter.

The resulting backend is assigned to the notebook-level variable `index`, allowing the remaining RAG workflow to operate independently of how the documents are stored and searched.

In [2]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

## 3. Test the existing RAG pipeline

First, create the course teaching assistant used in the earlier RAG examples.

In [3]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

Ask the same question twice: first with the correct spelling of **Ollama**, and then with a deliberate typo.

In [4]:
print(assistant.rag("How do I run Ollama locally?"))

To run Ollama locally:

1. Install Ollama from https://ollama.com/download  
   - macOS: download the `.pkg`
   - Windows: download the `.msi`
   - Linux: run:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. In a terminal, start a local model:
   ```bash
   ollama run llama3
   ```
   This downloads the LLaMA 3 model, starts it locally, and opens a chat-like interface.

3. To check the local server:
   ```bash
   curl http://localhost:11434
   ```
   You should get a response like:
   ```json
   {"models": [...]}
   ```

If you want to use it from Python, install the client with:

```bash
pip install ollama
```

and then use:

```python
import ollama

response = ollama.chat(
    model='llama3',
    messages=[{"role": "user", "content": your_prompt}]
)

print(response['message']['content'])
```


In [5]:
print(assistant.rag("How do I run Olama locally?"))

The FAQ doesn’t mention **Olama** specifically, but it does say you can run the course **locally** instead of in Codespaces if you’re comfortable setting up the needed tools.

To run locally, set up:

- Python
- `uv`
- Jupyter
- Docker
- any other tools needed for the module

Also, if you run locally, make sure you:

- document your setup
- keep your environment reproducible

If you meant **Ollama** specifically, I don’t have any FAQ entry for how to run it locally.


### Limitation: retrieval can be sensitive to spelling

The misspelled query retrieves less relevant context. This illustrates that the retrieval stage may be sensitive to spelling and query phrasing, even though the language model itself could understand the intended question.

## 4. Let the model formulate the search query

Instead of having the RAG assistant pass the user’s query directly to the index, we can expose the application-owned `search()` function to the language model as a tool. We do this by sending `search_tool`—a JSON-compatible definition describing the function and its arguments—to the model. The model can then formulate a more suitable search query—for example, by correcting a spelling error—and return a structured request asking our application to execute `search()`. The application executes the function and returns the retrieved FAQ entries to the model, which uses them to produce a grounded answer.

Before adding the tool, send the enrollment question to the model without FAQ context to establish a baseline.

Example of using the OpenAI Responses API to get a response from the model, _without_ the FAQ context included in the prompt:


In [6]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

print(response.output_text)

Possibly — but it depends on the course’s enrollment policy and whether registration is still open.

If you want, I can help you figure it out by checking:
- the course name
- the institution/platform
- whether it’s live, self-paced, or semester-based
- any deadline or enrollment page you found

If you mean “Can I still join after it started?”, tell me the course details and I’ll help you assess your chances and draft a message to the instructor or support team.


Without FAQ context, the model can only give a general answer such as "it depends on the course" or "check the course website." It does not know the course-specific enrollment and certificate rules, which is why we need retrieval.

### Define the application-owned search function

The following function duplicates the retrieval logic in `RAGBase`. It searches the index directly and returns the five most relevant LLM Zoomcamp FAQ entries.

`search()` accepts only `query`, because that is the only argument the model should supply. The function refers to the notebook-level `index` created in Section 2 rather than receiving it as an argument. Python resolves this global name when `search()` runs.

This keeps the tool interface simple for the teaching example. In a larger application, we would normally make the index dependency explicit—for example, by using a class or a factory function that receives the index.

In [7]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    # the search function returns the top 5 results from the index based on the query, 
    # with boosting and filtering applied.
    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

### Describe the function to the model

Here, **the model** means the remote language model selected in the API request (`gpt-5.4-mini` below), not the `openai_client` Python object. The client only sends the request and receives the response.

The model does not see the Python implementation of `search()`. Instead, we create `search_tool`, a JSON-compatible tool definition describing what the function does and which arguments it accepts. In the next section, our code passes that definition through `openai_client` to the selected model using `tools=[search_tool]`.

The tool definition is independent of the language used to implement the function. The same JSON Schema can describe a tool implemented in Python, TypeScript, Java, or another language.

In [8]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": (
        "Search the LLM Zoomcamp FAQ for information relevant to the "
        "user's question. Use a concise search query, correcting obvious "
        "spelling errors while preserving important technical terms."
    ),
    "strict": True,
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": (
                    "Concise query to search for in the course FAQ. Correct obvious "
                    "spelling errors and preserve important technical terms."
                )
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

The tool name, description, and parameter schema jointly tell the selected language model when and how to request the function. Clear descriptions improve tool selection, while the JSON Schema constrains the expected arguments. Here, `query` is required and additional properties are not allowed.

Setting `strict=True` asks the API to enforce the schema rather than treating it as best-effort guidance. Strict mode requires every property to be listed in `required` and each object to set `additionalProperties=False`; this schema satisfies both conditions. Defining `search_tool` still does not contact the API or execute `search()`.

At this point, the model has not seen the FAQ contents. It formulates the query from the user's question, the tool description, and its general ability to identify intent and choose useful search terms. For example, it can reformulate “I just discovered the course. Can I join it?” as a concise query about joining a course late. It does not need to know which FAQ entries exist; discovering matching entries is the job of `search()`.

The model is therefore doing linguistic work—identifying intent, correcting spelling, and choosing keywords—not retrieving facts or answering from the FAQ. The factual context arrives only after the application executes `search()` and sends the FAQ results back in the second API request.

### Request a tool-assisted answer

Send the same question again. `openai_client.responses.create(...)` sends the request, `model="gpt-5.4-mini"` selects the language model that reasons about it, and `tools=[search_tool]` tells that model which application-owned tool is available.

By default, tool selection is automatic: the model may either request a tool or answer directly. This tutorial's later cells expect a `function_call`, so `tool_choice` explicitly requires the model to request `search`. We force the tool only for this first request; the continuation requests omit `tool_choice` so the model remains free to produce a final answer after it receives the search results.

In [12]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
    tool_choice={"type": "function", "name": "search"},
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"join course late can I join it enrollment registration course start FAQ"}', call_id='call_aE3nSdzf1Hnzkiqyy0g4b22B', name='search', type='function_call', id='fc_01dc103096141d34006a69cd294a9481a38d5f92c182ddfa4d', namespace=None, status='completed')]

Because this request explicitly selected `search` with `tool_choice`, the response contains an item whose type is `function_call` instead of a final answer. This is a structured request for our application to run the search function using the arguments provided.

The generated arguments contain a search query related to late enrollment. Your exact query may differ because model output is not deterministic.

Notice that the model does not necessarily copy the user's question verbatim. It can reformulate the question into terms that may work better for retrieval.

Inspect the generated tool call, its JSON-encoded arguments, and the `call_id` that will connect the search result to this request.

In [13]:
response.id

'resp_01dc103096141d34006a69cd28c05481a394d801169a7ea830'

In [14]:
response.output[0]

ResponseFunctionToolCall(arguments='{"query":"join course late can I join it enrollment registration course start FAQ"}', call_id='call_aE3nSdzf1Hnzkiqyy0g4b22B', name='search', type='function_call', id='fc_01dc103096141d34006a69cd294a9481a38d5f92c182ddfa4d', namespace=None, status='completed')

In [15]:
response.output[0].arguments

'{"query":"join course late can I join it enrollment registration course start FAQ"}'

In [16]:
response.output[0].call_id

'call_aE3nSdzf1Hnzkiqyy0g4b22B'

In [17]:
response.output[0].name

'search'

In [18]:
response.output[0].type

'function_call'

`status="completed"` means the model finished generating the tool-call request. It does **not** mean that the search function has run.

The model does *not* execute this Python function. It only asks our application to execute it.

## 5. Execute the function and return its result

Two different identifiers are used here, at different levels:

- `previous_response_id` links the **API requests**: it tells the model to continue from the earlier response, preserving the conversation context.
- `call_id` links the **tool messages**: it matches one specific `function_call` from the model with the corresponding `function_call_output` returned by our application.

They are complementary, not interchangeable. In Option A we need both: `previous_response_id` restores the earlier response context, while `call_id` says which tool call the returned result belongs to.

The key pieces are:

- `search()` — the executable Python retrieval function owned by our application.
- `search_tool` — the JSON schema that tells the model that `search()` exists and which arguments it accepts. Passing this schema does not execute a search.
- `function_call_output` — supplies the result produced by `search()` back to the model.

In this single-tool example, the following cell contains the **only actual retrieval call**. The later API calls return this same result to the model; they do not perform an augmented search.

The flow is:

```text
User question
    ↓
Model receives the search_tool schema
    ↓
Model returns a function_call request
    ↓
Application executes search(**args)  ← the only search; args contains the model's requested query
    ↓
Application returns function_call_output, which is passed to the model in serialized format
    ↓
Model answers or requests another tool call
```

Our application must:

1. parse the generated arguments;
2. call the corresponding Python function;
3. serialize its result; and
4. return that result using the same `call_id`.

In [ ]:
import json

call = response.output[0]

# Function arguments are returned as a JSON-encoded string.
args = json.loads(call.arguments)
print(args)  # the query suggested by the model to pass to search()

# Custom functions run in our application, not inside the OpenAI API.
# **args unpacks the generated arguments and passes them to search().
results = search(**args)  # The only retrieval call in this example.
print(results)  # the search results returned by our custom function in response to the model's query

# 'results' is a Python list of dictionaries; serialize it as JSON text.
# indent=2 only makes that text easier for humans to read.
result_json = json.dumps(results, indent=2)

{'query': 'join course late can I join the course after it started'}
[{'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}, {'id': '04919992b3', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'How should I start the course and follow the weekly workflow?', 'answer': 'Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).\n\nYou can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed in the [course management platform](https://courses.datatalks.club/llm-zo

In [ ]:
print(result_json)  # the search results serialized as JSON text,
                    # which can be passed back to the model in a subsequent request for further processing.

[
  {
    "id": "74eb249bbf",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still join?",
    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."
  },
  {
    "id": "04919992b3",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "How should I start the course and follow the weekly workflow?",
    "answer": "Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).\n\nYou can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed in the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).

In [34]:
# call_id = unique identifier for this function-call request.
# It links the model's request to the result returned by our application.

call.call_id    

'call_ZvO2Y56ilZKoS6KOGZfh9mXp'

In [36]:
type(results)

list

In [38]:
type(result_json)

str

`results` and `result_json` contain the same search-result data, but they are different Python types:

- `results` is a Python `list` containing `dict` objects, returned directly by `search()`.
- `result_json` is a `str` created by serializing that Python structure with `json.dumps()`.

The newlines and spaces come from `indent=2`; they only make the JSON text easier to read. `json.dumps(results)` without `indent` would produce a more compact string containing the same data. The important operation is serialization, because a Responses API `function_call_output` is typically supplied as a string—JSON in this example, although plain text is also allowed.

The previous cell uses `print(result_json)`, so the notebook renders the string with real line breaks and without surrounding quotes. If `result_json` were instead evaluated directly as the cell's final expression, Jupyter would display its Python representation, including surrounding quotes and escaped sequences such as `\n`.

### Option A: continue with `previous_response_id`

This call does **not** run `search()` again. It sends the already computed `result_json` back to the model:

- `previous_response_id=response.id` continues from the entire earlier model response, which contains the conversation context and the tool request.
- `call_id=call.call_id` matches `result_json` to the particular `function_call` that requested it. This is especially important when a response contains more than one tool call.
- `tools=[search_tool]` makes the schema available again in case the model decides another search is necessary. It does not execute the tool.

Put another way: `previous_response_id` answers **“Which prior response are we continuing?”**, while `call_id` answers **“Which tool call does this output satisfy?”**

In this continuation request, the model can read `result_json`, so it now has the retrieved FAQ context that it did not have when formulating the query. It judges whether the results are sufficient by checking whether they address the user's question and contain enough information to compose an answer.

When the supplied FAQ results appear sufficient, the model returns a final text answer. If they appear insufficient or unclear, the model can return another `function_call`, potentially with a revised query; the application must execute that new call and continue the loop.

This sufficiency decision is a model judgment, not a guarantee that the best FAQ entry was retrieved or that the FAQ is complete. Production applications commonly add safeguards such as relevance thresholds, explicit instructions to search again when evidence is inadequate, citations to retrieved entries, and a limit on repeated tool calls.

In [39]:
final_response = openai_client.responses.create(
    model="gpt-5.4-mini",
    previous_response_id=response.id,   # the response from the model that suggested the search query
    # Re-advertise the schema; this does not execute search().
    tools=[search_tool],
    input=[
        {
            "type": "function_call_output",
            "call_id": call.call_id,    # Link this output to the model's function-call request.
            "output": result_json,
        }
    ],
)

print(final_response.output_text)

Yes — you can still join.

You can start anytime, and the videos/materials are available. If you want a certificate, though, you’ll need to submit your project while submissions are still open.


### Option B: manage and replay the history manually

This is an alternative to Option A, not a second retrieval method. Instead of using `previous_response_id`, the application manually replays the original question, the model's function-call request, and the same `result_json` produced above.

In [40]:
response.output

[ResponseFunctionToolCall(arguments='{"query":"join course late can I join the course after it started"}', call_id='call_ZvO2Y56ilZKoS6KOGZfh9mXp', name='search', type='function_call', id='fc_0f2969f20afa59c2006a67426e02148191bd04a7b942d0c327', namespace=None, status='completed')]

In [41]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'}]

In [42]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    # Link this result to the model's function-call request.
    "call_id": call.call_id,
    "output": result_json,
})

The model can now see both what it requested and what the function returned. `extend()` adds every item from the `response.output` list, while `append()` adds the single function-result item.

In [43]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course late can I join the course after it started"}', call_id='call_ZvO2Y56ilZKoS6KOGZfh9mXp', name='search', type='function_call', id='fc_0f2969f20afa59c2006a67426e02148191bd04a7b942d0c327', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_ZvO2Y56ilZKoS6KOGZfh9mXp',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "04919992b3",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "How should I start the course and follow the weekly workflow?",\n    "answer": "Star

Replay the complete history to obtain the final answer:

In [44]:
manual_final_response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

print(manual_final_response.output_text)

Yes — you can still join the course.

If you want a certificate, make sure to submit your project while submissions are still open.


The model now has the original question, its own request to call `search`, and the FAQ results. It can therefore produce a course-specific answer.

Only `search(**args)` performs retrieval in this example. `search_tool` describes the available function, while the subsequent Responses API call supplies its output to the model.

Each new request must be connected to the previous interaction. We can either pass `previous_response_id`, as in Option A, or replay the relevant history ourselves, as in Option B. Both approaches use the same search result.

This is the complete function-calling loop for a single tool call. Compared with a one-call RAG pipeline, the model-driven tool loop requires at least two model requests: one to request the tool and another to interpret its result.

This pattern is commonly described as **agentic RAG**, **tool use**, or **function calling**. In a general tool-using workflow, the model may decide whether and how to request an available tool. In this teaching example, `tool_choice` requires a search call, while the model decides how to formulate its query.

> **Production note:** This notebook assumes that the response contains exactly one tool call. Production code should inspect every item in `response.output`, execute each `function_call`, and continue the loop until the model returns a final message rather than another tool request.

## 6. Token usage and cost

Option A made two API calls: `response` generated the tool request, and `final_response` interpreted the search result. To calculate the cost of the complete tool-assisted turn, add the usage from both responses.

The manually replayed alternative makes another final-answer request when executed, so treat it as a separate demonstration rather than part of the Option A total.

In [45]:
responses = [response, final_response]

input_tokens = sum(item.usage.input_tokens for item in responses)
cached_input_tokens = sum(
    item.usage.input_tokens_details.cached_tokens for item in responses
)
output_tokens = sum(item.usage.output_tokens for item in responses)

input_tokens, cached_input_tokens, output_tokens

(902, 0, 72)

In [47]:
# Prices per million tokens for gpt-5.4-mini.
# Verify current prices before using this calculation in production.
def calculate_gpt54mini_price(input_tokens, cached_input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.75
    CACHED_INPUT_PRICE_PER_MILLION = 0.075
    OUTPUT_PRICE_PER_MILLION = 4.50

    uncached_input_tokens = input_tokens - cached_input_tokens
    input_cost = (uncached_input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    cached_input_cost = (
        cached_input_tokens / 1_000_000
    ) * CACHED_INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + cached_input_cost + output_cost

    return {
        "input_cost": input_cost,
        "cached_input_cost": cached_input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

cost = calculate_gpt54mini_price(
    input_tokens=input_tokens,
    cached_input_tokens=cached_input_tokens,
    output_tokens=output_tokens,
)
print("Total cost: $", round(cost["total_cost"], 8))

Total cost: $ 0.0010005
